# Actor-Critic Method

## Value Network and Policy Network

 `State-value function`：
$$
V_\pi (s) = \sum_a \pi (a|s) \cdot Q_\pi (s, a).
$$

`policy network` (actor)：
- 使用神经网络$\pi(a|s;\theta)$来近似拟合$\pi(a|s)$.
- $\theta$为可训练的神经网络参数.

`value network`(critic)：
- 使用神经网络$q(s, a; w)$来近似拟合$Q_\pi(s, a)$.
- $w$为可训练的神经网络参数.

## Train the networks

使用策略网络近似策略函数，使用价值网络近似动作价值函数；如此一来，状态价值函数$V$可以使用$\theta、w$来近似。
$$
V(s; \theta, w) = \sum_a \pi(a|s;\theta) \cdot q(s, a; w)
$$
训练时要更新$\theta$和$w$两个参数，但更新策略不同：
- 更新策略网络$\pi (a|s;\theta)$，需要是的状态价值函数$V(s;\theta, w)$的值更大；
- 更新价值网络$q(s, a; w)$，通过环境给出的`rewards`更新评价（网络参数开始是初始化的，即最开始的评价是乱猜的，网络需要更新参数使得给出分出贴近环境给出的`reward`，但`reward`是需要一轮游戏结束后才全部公开）。

具体步骤：
- 观测到状态$s_t$；
- 根据策略网络$\pi(\cdot|s_t;\theta+t)$随机抽样行为$a_t$；
- 做出行为$a_t$，观察新状态$s_{t+1}$和奖励$r_t$；
- 使用`TD`算法更新价值网络参数$w$；
- 使用策略梯度算法更新策略网络参数$\theta$。

### update value network $q$ using `TD`

- 计算$q(s_t, a_t; w_t)$和$q(s_{t+1}, a_{t+1}; w_t)$（两个动作的评分）；
- 计算`TD target`：$y_t = r_t + \gamma \cdot q(s_{t+1}, a_{t+1}; w_t)$；
- 计算损失：$L(w) = \frac 1 2 [q(s_t, a_t; w) - y_t] ^ 2$；
- 梯度下降：$w_{t+1} = w_t - \alpha \cdot \frac {\partial L(w)}{\partial w} | w=w_t$。

### update policy network $\pi$ using `policy gradient`

- 通过神经网络拟合状态价值函数：$V(s; \theta, w) = \sum_a \pi(a|s; \theta) \cdot q(s, a; w)$
- 策略梯度：$g(a, \theta) = \frac {\partial log \pi (a|s, \theta)}{\partial \theta} \cdot q(s_t, a; w)$，而上一章得出$\frac {\partial log \pi (a|s, \theta)}{\partial \theta} = \mathbb E_A[g(a, \theta)]$
    - 既然$g(A, \theta)$函数是对期望的无偏估计，直接用一个$g(A, \theta)$即可，即对期望的蒙特卡洛近似。
    - 根据策略网络随机抽样$a \sim \pi(\cdot | s_t; \theta_t)$；
    - 随机梯度上升：$\theta_{t+1} = \theta_t + \beta \cdot g(a, \theta_t)$